# Claude Code in Action — Live Walkthrough

This is an instructor walkthrough notebook. Each section pairs lecture markdown with a runnable Anthropic SDK demo so we can talk through Claude Code concepts and watch them play out via the API in real time.

## Agenda

1. What is a Coding Assistant?
2. Claude Code in Action
3. Adding Context
4. Making Changes
5. Controlling Context
6. Custom Commands
7. Extending Claude Code with MCP Servers
8. GitHub Integration
9. Introducing Hooks
10. Defining Hooks
11. Implementing a Hook
12. Useful Hooks
13. The Claude Code SDK

---
## Setup

1. Create a `.env` file at the project root containing:
   ```
   ANTHROPIC_API_KEY=sk-ant-...
   ```
2. Add `.env` to your `.gitignore` so the key never lands in source control.
3. Install the SDK and helpers (uncomment the next cell to run).

In [ ]:
# %pip install -q anthropic python-dotenv

The next cell loads `ANTHROPIC_API_KEY` from `.env`, builds the SDK client, and prints a three-line sanity check so a missing key fails loudly *before* any API call.

In [ ]:
import os
import anthropic
from dotenv import load_dotenv

load_dotenv()

client = anthropic.Anthropic()
model = "claude-sonnet-4-6"

print(f"SDK version: {anthropic.__version__}")
print(f"Model:       {model}")
print(f"Key loaded:  {bool(os.environ.get('ANTHROPIC_API_KEY'))}")

### Shared helpers

Every demo in this notebook uses the same three helpers:

- `add_user_message(messages, text)` — append a user turn to a running conversation.
- `add_assistant_message(messages, text)` — append an assistant turn (useful for replaying transcripts or pre-filling).
- `chat(messages, system=..., temperature=..., stop_sequences=...)` — single-call wrapper around `client.messages.create()` that returns the assistant text.

Defining them once means later demos stay tiny: each one varies a single argument, and the focus stays on the concept being demonstrated.

In [ ]:
def add_user_message(messages, text):
    messages.append({"role": "user", "content": text})
    return messages

def add_assistant_message(messages, text):
    messages.append({"role": "assistant", "content": text})
    return messages

def chat(messages, system=None, temperature=1.0, stop_sequences=None):
    """Send messages to Claude and return the assistant text."""
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
    }
    if system is not None:
        params["system"] = system
    if stop_sequences is not None:
        params["stop_sequences"] = stop_sequences
    response = client.messages.create(**params)
    return response.content[0].text

---
# 1. What is a Coding Assistant?

A coding assistant is a tool that wraps a language model so it can carry out development work — reading source files, running commands, writing patches, and verifying results.

### The core loop

| Step | Who acts | What happens |
|---|---|---|
| 1 | User | Hands the assistant a task ("fix this stack trace"). |
| 2 | Assistant + LM | LM gathers context by asking the assistant to read files. |
| 3 | LM | Forms a plan from what it learned. |
| 4 | Assistant + LM | LM emits formatted action requests; assistant executes them. |

### Why "tool use" sits at the heart of this

A language model only ingests text and emits text. It cannot, on its own, open a file or run `pytest`. The assistant brokers the difference: it appends instructions like *"to read a file, respond with `READ_FILE: <path>`"*, parses that response, performs the real action, and feeds the result back to the model.

### Why Claude leads here

Claude models are tuned hard on tool use — choosing the right tool, chaining several of them, and recovering from a failed call. That is what makes Claude Code feel less like an autocomplete and more like a teammate that finishes the job.

### Demo: simulating the tool-use loop by hand

We give Claude a system prompt that defines two pretend "tools" (`READ_FILE` and `RUN_TESTS`) and a dummy bug report. The model should respond with one of those formatted actions instead of trying to fix the bug from thin air. Watch for the exact format Claude returns — that is the protocol the real Claude Code parses every turn. Swap the bug description live to see how the choice of action changes.

In [ ]:
system = (
    "You are a coding assistant. You cannot read files or run code directly. "
    "To take an action, respond with exactly one line in one of these formats:\n"
    "  READ_FILE: <path>\n"
    "  RUN_TESTS: <test_path>\n"
    "Do not explain. Output only the action line."
)

bug_report = (
    "Tests in tests/test_auth.py are failing with `AttributeError: 'NoneType' "
    "object has no attribute 'token'`. Decide what to do first."
)

messages = add_user_message([], bug_report)
print(chat(messages, system=system, temperature=0))

> **🏫 During class:**
> 1. Run the cell once with the auth bug report — Claude should emit `READ_FILE: tests/test_auth.py`.
> 2. Say out loud: *"That single line is the entire trick. Real Claude Code parses this exact pattern, runs the file read for it, and feeds the contents back."*
> 3. Edit `bug_report` to *"Tests are passing locally but failing in CI. Decide what to do first."* and re-run — point out how the action shifts to `RUN_TESTS` because the model picks a different opening move.

---
# 2. Claude Code in Action

Claude Code is the AI coding assistant built around Claude. It ships with a default toolbox — file read/write, command execution, basic dev ops — and accepts new tools through MCP servers and config.

### Things it has actually shipped

- **Performance work.** Profiled the Chalk JS library (~429M weekly downloads), found bottlenecks, applied fixes — 3.9× throughput improvement.
- **Data analysis.** Iterated through churn analysis on a streaming-platform CSV inside Jupyter, viewing each cell's output before deciding the next step.
- **Browser-driven UI work.** With the Playwright MCP server attached, opened a browser, screenshot-checked styling, and refined the design loop.
- **GitHub Actions reviews.** Caught a PR that piped a user email from a Lambda into an S3 bucket shared with an external partner — a PII leak the human reviewer had missed.

### The takeaway

Claude Code is a flexible assistant that grows with the team. New tool, new capability — no model retraining required.

### Demo: a tiny code-review pass via the API

We hand Claude a short Python function with a real bug (off-by-one in pagination math) and ask for a focused review. This is the same shape of work Claude Code does in a PR review action — stripped down to a single API call so we can watch the reasoning. Try replacing the snippet with bug-free code to see whether Claude pushes back instead of inventing problems.

In [ ]:
snippet = '''
def paginate(items, page, page_size):
    start = page * page_size
    end = start + page_size
    return items[start:end]

# called as: paginate(items, page=1, page_size=10) for the "first page"
'''

review_prompt = (
    "Review this Python function. Identify any bug, explain the impact in one "
    "sentence, and propose a one-line fix. Be concise.\n\n"
    f"```python{snippet}```"
)

messages = add_user_message([], review_prompt)
print(chat(messages, temperature=0.2))

> **🏫 During class:**
> 1. Run the review cell — Claude should flag that callers passing `page=1` skip the first page (1-based vs 0-based mismatch).
> 2. Say: *"That's a one-shot review. Claude Code does this same thing inside a GitHub Action, but with the full file tree and prior comments as context."*
> 3. Replace the snippet with a correct version and rerun — ask the room: *"Did Claude invent a problem, or did it actually back off?"* This is the same calibration question we'd ask of any reviewer.

---
# 3. Adding Context

The biggest lever on assistant quality is context — what the model knows about *this* codebase right now. Too little, and it hallucinates conventions. Too much, and signal drowns in noise.

### How Claude Code manages it

| Mechanism | What it does |
|---|---|
| `/init` | Scans the repo on first run and writes a `CLAUDE.md` summarizing project layout, key files, conventions. Included in every later request. |
| Project `CLAUDE.md` | Team-shared, committed to source control. |
| Local `CLAUDE.md` | Per-developer notes, gitignored. |
| Machine-level `CLAUDE.md` | Global preferences across all projects. |
| `#` shortcut (memory mode) | Edit `CLAUDE.md` files in plain English — *"#always run pytest after edits"*. |
| `@filename` | Pin a specific file into the request so Claude doesn't have to search for it. |

### Why it matters

A good `CLAUDE.md` makes the difference between Claude knowing your DB schema lives in `db/schema.prisma` versus it grepping around for ten seconds and possibly editing the wrong file.

### The API analogue

On the raw API, the `system` prompt plays the role of `CLAUDE.md`: a place to drop conventions, schema snippets, and standing instructions that ride along with every turn.

### Demo: same question, with and without project context

We ask Claude to write a function call. The first call uses no system prompt; the second passes a tiny "project conventions" system prompt — the kind of thing a real `CLAUDE.md` would carry. The second answer should match the conventions exactly. Tweak the conventions and rerun to see Claude adapt with no other change.

In [ ]:
question = "Write a one-line example call to our `fetch_user` function."

print("--- No context ---")
print(chat(add_user_message([], question), temperature=0))

project_conventions = (
    "Project conventions:\n"
    "- All async functions are awaited inside `with trace_span(...)` blocks.\n"
    "- `fetch_user` is async and takes a single keyword arg `user_id: UUID`.\n"
    "- We never use positional arguments in this codebase."
)

print("\n--- With project context ---")
print(chat(add_user_message([], question), system=project_conventions, temperature=0))

> **🏫 During class:**
> 1. Run the cell once and read both outputs aloud back-to-back.
> 2. Say: *"This is exactly what `CLAUDE.md` does — it preloads the conventions so Claude doesn't have to guess. The system prompt is the API equivalent."*
> 3. Add one bogus rule to `project_conventions` (e.g., *"All function names are SCREAMING_SNAKE_CASE"*) and rerun — ask: *"Where does Claude follow the rule, and where does its training data win?"*

---
# 4. Making Changes

Claude Code edits land through a small set of techniques worth memorizing.

### The toolbox

- **Screenshot context.** `Ctrl-V` (not `Cmd-V` on macOS) pastes an image into the prompt — useful for *"this button is misaligned, fix it"*.
- **Plan Mode** (`Shift+Tab` twice). Forces Claude to scout files and write an implementation plan before editing. Wins on **breadth**.
- **Thinking Mode** ("ultrathink", "think hard", etc.). Allocates a larger reasoning budget. Wins on **depth** — gnarly logic, subtle bugs.
- **Both at once.** Real-world tasks often deserve a Plan + Thinking combo. Tokens cost more; outcomes improve.
- **Git integration.** Claude stages, commits, and writes its own commit messages.

### Plan vs Think — quick rule

| Symptom | Reach for |
|---|---|
| Touches many files, not sure where | Plan |
| One file, but the logic is twisty | Think |
| Both | Both |

### The API analogue

Plan/Think both surface as the `thinking` parameter in the SDK — the model gets an explicit reasoning budget before it answers.

### Demo: extended thinking on a tricky logic puzzle

We pose a small probability puzzle and run it twice — once with no thinking budget, once with extended thinking enabled. The second call exposes a `thinking` content block before the final answer. The takeaway isn't always *"the answer changes"* — sometimes it's *"the answer arrives with visible reasoning the first call had to do silently"*. Try a harder problem live and compare confidence.

In [ ]:
puzzle = (
    "A bag holds 3 red, 4 blue, 5 green marbles. You draw three without "
    "replacement. What is the probability all three are different colors? "
    "Give the exact fraction in lowest terms and a one-line justification."
)

print("--- No extended thinking ---")
print(chat(add_user_message([], puzzle), temperature=0))

print("\n--- With extended thinking ---")
response = client.messages.create(
    model=model,
    max_tokens=2000,
    thinking={"type": "enabled", "budget_tokens": 1500},
    messages=[{"role": "user", "content": puzzle}],
)
for block in response.content:
    if block.type == "thinking":
        print("[thinking, first 300 chars]:", block.thinking[:300], "...\n")
    elif block.type == "text":
        print("[answer]:", block.text)

> **🏫 During class:**
> 1. Run the cell once. Read the no-thinking answer, then the thinking answer, side by side.
> 2. Say: *"Plan Mode is breadth — go scout the repo. Thinking Mode is depth — chew on this hard step. The API exposes the depth lever directly."*
> 3. Swap the puzzle for a harder one (e.g., a Project Euler problem) and ask: *"Does the no-thinking answer get worse, or just less explained?"* — surfaces that thinking isn't always about correctness, sometimes about visibility.

---
# 5. Controlling Context

Long sessions accumulate noise. Claude Code gives you four levers to keep the working set tight.

| Action | Effect | Use when |
|---|---|---|
| `Esc` (single) | Interrupts Claude mid-response. | You see Claude going down the wrong path. |
| `Esc` + `#` | Stop, then write a memory note so the mistake doesn't repeat. | Same wrong move twice in a row. |
| `Esc` `Esc` (double) | Rewinds to an earlier point in the conversation. | Long debug detour landed somewhere useful — jump back, drop the detour. |
| `/compact` | Summarizes the conversation while preserving learned task context. | Claude has built up real expertise but the transcript is cluttered. |
| `/clear` | Wipes the conversation entirely. | Switching to an unrelated task. |

### The API analogue

There is no magic — context is just the `messages` list you send. Trimming, summarizing, and rewinding are all things you do to that list yourself.

### Demo: summarizing history before a follow-up

We build a small multi-turn conversation, then ask Claude to summarize it into a single user message. Round two replays the *summary* instead of the full transcript and asks a follow-up. This is exactly what `/compact` does inside Claude Code, with the levers exposed. Watch the token cost drop without losing the task knowledge.

In [ ]:
history = []
add_user_message(history, "I'm building a Flask API for a todo app.")
add_assistant_message(history, "Got it — what storage are you using?")
add_user_message(history, "SQLite for now, Postgres later. Single-user, no auth.")
add_assistant_message(history, "Understood. We'll keep models simple and avoid premature auth scaffolding.")
add_user_message(history, "Cool. Endpoints: list, create, complete, delete.")
add_assistant_message(history, "Standard CRUD. I'll suggest REST routes when you're ready.")

summary_prompt = (
    "Summarize the conversation above into a single short paragraph capturing "
    "every decision and constraint. No greetings, no filler."
)
summary = chat(history + [{"role": "user", "content": summary_prompt}], temperature=0)
print("--- Compacted summary ---\n", summary, "\n")

compacted = []
add_user_message(compacted, f"Project context: {summary}")
add_user_message(compacted, "Now write the Flask route for `POST /todos`.")
print("--- Follow-up using compacted context ---\n", chat(compacted, temperature=0.2))

> **🏫 During class:**
> 1. Run the cell. Show the summary first, then the follow-up output.
> 2. Say: *"Compact is summarize-then-replace. Same task knowledge, smaller transcript, fewer tokens."*
> 3. Ask the room: *"What did the summary drop that we might regret later?"* — surfaces that compaction is lossy and you choose what survives.

---
# 6. Custom Commands

Custom commands are user-defined slash commands. They live in `.claude/commands/` as plain markdown files; the filename becomes the command name (`audit.md` → `/audit`).

### Anatomy

- **Body**: free-form markdown — instructions for what Claude should do.
- **`$arguments`**: placeholder substituted at runtime. `/audit src/auth.ts` swaps `src/auth.ts` into `$arguments`.
- **Activation**: restart Claude Code after creating the file.

### What they're for

Anything you'd otherwise paste a paragraph for, every time:

- `/audit <file>` — dependency / vulnerability check.
- `/test <function>` — generate tests for one symbol.
- `/regen <component>` — regenerate UI from a screenshot.

### The API analogue

A custom command is just a parameterized prompt template. The same idea works in raw API calls — keep the template in code, swap arguments in.

### Demo: a parameterized prompt template, like an `/audit` command

We define a Python function that mimics the `audit.md` command — a fixed instruction body with a single `$arguments`-style slot. Calling it with different snippets feels exactly like running `/audit <thing>` inside Claude Code. The example we hand it is a textbook SQL-injection antipattern (raw string concatenation of untrusted input into a query); Claude should flag it as HIGH severity. Try defining a second template (`/explain`, `/refactor`) live to drive home that commands are just templates with arguments.

In [ ]:
AUDIT_TEMPLATE = (
    "You are a security-focused code auditor. Examine the following code for "
    "injection risks, unsafe input handling, or other vulnerabilities. Return "
    "a bulleted list with severity tags (LOW/MED/HIGH). If nothing is wrong, "
    "say so.\n\n"
    "```\n$arguments\n```"
)

def slash_audit(arguments: str) -> str:
    prompt = AUDIT_TEMPLATE.replace("$arguments", arguments)
    return chat(add_user_message([], prompt), temperature=0)

# Example: a textbook unsafe pattern — string concatenation into SQL.
dangerous = '''
def find_user(conn, name_from_form):
    query = "SELECT * FROM users WHERE name = '" + name_from_form + "'"
    return conn.execute(query).fetchall()
'''

print(slash_audit(dangerous))

> **🏫 During class:**
> 1. Run `slash_audit(dangerous)` — Claude should flag SQL injection as HIGH and suggest parameterized queries.
> 2. Say: *"This is a custom command in eight lines of Python. The Claude Code version is the same idea — a markdown file, a placeholder, a slash to invoke it."*
> 3. Ask the room: *"What command would you build for your repo this week?"* — collect 2–3 ideas live.

---
# 7. Extending Claude Code with MCP Servers

MCP servers are external tool providers — programs Claude Code talks to over a protocol to gain new capabilities. They run locally or remotely.

### Anatomy

- **Install**: `claude mcp add <name> <start-command>` registers a server.
- **First use**: Claude Code prompts for permission per tool.
- **Auto-approve**: add `MCP__<servername>` to the `allow` array in `.claude/settings.local.json`.

### Real example: Playwright MCP

Claude opens `localhost:3000`, takes a screenshot, generates a UI component, screenshots again, compares, refines its prompt — a full visual feedback loop with no human in the middle.

### The API analogue

At the API level, MCP tools land in the same `tools=[...]` array as any other tool. Same JSON schema, same `tool_use` content blocks. The MCP protocol is the *plumbing*; the model interface is unchanged.

### Demo: tool use with the SDK — the same shape MCP servers expose

We define a single fake `take_screenshot` tool, send a request that obviously needs it, and inspect what Claude returns. The interesting bit is not the answer — it's the `tool_use` content block. That block is the same wire format Claude emits whether the tool comes from a built-in or an MCP server. Try changing the user message to something that *doesn't* need the tool to see Claude skip it.

In [ ]:
tools = [
    {
        "name": "take_screenshot",
        "description": "Capture a screenshot of the given URL and return PNG bytes.",
        "input_schema": {
            "type": "object",
            "properties": {"url": {"type": "string"}},
            "required": ["url"],
        },
    }
]

response = client.messages.create(
    model=model,
    max_tokens=400,
    tools=tools,
    messages=[{"role": "user", "content": "Grab a screenshot of https://example.com and tell me what's on it."}],
)

print("stop_reason:", response.stop_reason)
for block in response.content:
    if block.type == "tool_use":
        print("tool requested:", block.name, "with input:", block.input)
    elif block.type == "text":
        print("text:", block.text)

> **🏫 During class:**
> 1. Run the cell — Claude should return `stop_reason='tool_use'` and request `take_screenshot` with `url="https://example.com"`.
> 2. Say: *"That JSON block is what an MCP server receives. The protocol does the running; the model only ever speaks tool-use JSON."*
> 3. Change the user message to *"What is 2+2?"* and rerun — show that Claude skips the tool entirely and answers directly. Tool use is opt-in, not mandatory.

---
# 8. GitHub Integration

Claude Code ships an official GitHub integration that lets it run inside GitHub Actions.

### Setup

1. `/install GitHub app` from inside Claude Code.
2. Install the Claude Code app on the repo or org.
3. Provide an API key.
4. Accept the auto-generated PR — it adds two GitHub Actions.

### Default actions

| Action | Trigger |
|---|---|
| Mention support | `@claude` in any issue or PR comment assigns work. |
| PR review | Automatic review on every new pull request. |

### Customization

Both actions are plain workflow files in `.github/workflows/`. You can:

- Inject custom instructions (project conventions, review focus).
- Attach MCP servers (e.g., Playwright for browser-based PR verification).
- **Permissions are explicit** — list every tool, including each MCP tool, individually. No shortcuts.

### Why it matters

A reviewer that runs on every PR catches the boring-but-deadly stuff (PII leaks, secrets, dependency surprises) before a tired human does.

### Demo: a "PR reviewer" prompt that posts via a `comment_pr` tool

We define a `comment_pr` tool and ask Claude to review a small diff. Claude responds with a `tool_use` block — the structured comment a real GitHub Action would forward to the GitHub API. The lesson: GitHub integration is just MCP-shaped tools wired to GitHub's REST endpoints. Replace the diff with a clean one to see Claude either skip the tool or post an LGTM comment.

In [ ]:
diff = (
    "diff --git a/handlers/user.py b/handlers/user.py\n"
    "+import logging\n"
    "+def handle(event):\n"
    "+    logging.info(f\"user_email={event['email']}\")\n"
    "+    return {'status': 'ok'}\n"
)

tools = [
    {
        "name": "comment_pr",
        "description": "Post a review comment on a GitHub pull request.",
        "input_schema": {
            "type": "object",
            "properties": {
                "severity": {"type": "string", "enum": ["info", "warn", "block"]},
                "body": {"type": "string"},
            },
            "required": ["severity", "body"],
        },
    }
]

system = "You are a strict PR reviewer. Flag PII or secrets. Use comment_pr exactly once."

response = client.messages.create(
    model=model,
    max_tokens=600,
    system=system,
    tools=tools,
    messages=[{"role": "user", "content": f"Review this diff:\n{diff}"}],
)

for block in response.content:
    if block.type == "tool_use":
        print(f"[{block.input['severity'].upper()}] {block.input['body']}")

> **🏫 During class:**
> 1. Run the cell — Claude should flag the email logging as `warn` or `block` PII.
> 2. Say: *"That tool call is what the GitHub Action sends to GitHub's REST API. The model never talks to GitHub directly — the wrapper does."*
> 3. Replace the diff body with a no-op refactor and ask the room: *"Will Claude still call the tool, or stay silent?"* — exposes whether the system prompt forces a comment or allows abstention.

---
# 9. Introducing Hooks

Hooks are commands that fire **before or after** Claude executes a tool. They are how you bolt deterministic checks onto a probabilistic assistant.

### Two flavors

| Hook | Fires | Can block? | Typical job |
|---|---|---|---|
| Pre-tool-use | Before the tool runs | Yes (exit 2) | Block reads of `.env`, deny destructive shell commands, gate writes outside `src/`. |
| Post-tool-use | After the tool runs | No | Auto-format on save, run `tsc --no-emit`, run tests, lint. |

### Where they live

In your settings file (global, project, or personal): `.claude/settings.local.json`. Each entry has a **matcher** (which tools it intercepts, e.g. `read|grep`) and a **command** (what to run).

### Why they're powerful

Anything you'd want to remind Claude about every turn — *"don't read .env"*, *"run the type checker after edits"* — graduates from a prompt nudge to an enforced rule.

### Demo: wrapping `chat()` with pre- and post-hooks

We build a tiny Python wrapper that simulates the hook lifecycle around a chat call: the pre-hook can refuse the call entirely, the post-hook can inspect the response. This is the same control flow Claude Code applies to tool calls. Watch the pre-hook block a forbidden phrase, then loosen the rule and watch the post-hook fire instead.

In [ ]:
class HookBlocked(Exception):
    pass

def chat_with_hooks(messages, pre_hook=None, post_hook=None, **kwargs):
    if pre_hook is not None:
        decision = pre_hook(messages)
        if decision == "block":
            raise HookBlocked("Pre-hook denied the call.")
    answer = chat(messages, **kwargs)
    if post_hook is not None:
        post_hook(answer)
    return answer

def pre_block_secrets(messages):
    last = messages[-1]["content"].lower()
    if ".env" in last or "api_key" in last:
        return "block"
    return "allow"

def post_log(answer):
    print(f"[post-hook] response length: {len(answer)} chars")

try:
    chat_with_hooks(
        add_user_message([], "Read the contents of my .env file."),
        pre_hook=pre_block_secrets,
        post_hook=post_log,
    )
except HookBlocked as e:
    print("BLOCKED:", e)

print("---")
print(chat_with_hooks(
    add_user_message([], "Tell me what hooks are useful for."),
    pre_hook=pre_block_secrets,
    post_hook=post_log,
))

> **🏫 During class:**
> 1. Run the cell — first call should print `BLOCKED:`, second should print a post-hook line plus the answer.
> 2. Say: *"Hooks turn fuzzy 'please don't' instructions into hard guardrails. The pre-hook is the bouncer, the post-hook is the auditor."*
> 3. Add `"password"` to the pre-hook's blocklist and ask the room: *"What else belongs here for your team?"*

---
# 10. Defining Hooks

Defining a hook means writing a small program that Claude Code will pipe tool-call data into.

### Implementation steps

1. Choose **pre** or **post**.
2. Pick the tools to monitor (matcher syntax: `read|grep`).
3. Read the JSON payload from **stdin**.
4. Decide based on the payload.
5. Exit with the right code.

### The payload (stdin, JSON)

```json
{
  "session_id": "...",
  "tool_name": "read",
  "tool_input": {"file_path": "/path/to/file"}
}
```

### Exit-code semantics

| Exit code | Meaning | Notes |
|---|---|---|
| 0 | Allow | Tool call proceeds. |
| 2 | Block | Pre-hook only; stderr becomes feedback to Claude. |
| any other | Error | Treated as failure. |

### A discoverability tip

Don't memorize tool names — ask Claude for the current list. They drift between releases; your memory won't.

### Demo: a hook in pure Python — same shape, no Claude Code needed

We write a function that takes the JSON payload Claude Code would send, inspects it, and returns an exit code plus a stderr message. Then we feed it three sample payloads. The point is to make the hook contract concrete: *"read JSON, decide, exit"* — that's it. Add a new rule (e.g., block destructive shell commands) live to show how easy it is to extend.

In [ ]:
import json

def env_guard(payload_json: str):
    payload = json.loads(payload_json)
    name = payload.get("tool_name", "")
    path = payload.get("tool_input", {}).get("file_path", "")
    if name in ("read", "grep") and ".env" in path:
        return 2, f"Blocked: refused to access {path} (path contains .env)"
    return 0, ""

samples = [
    '{"session_id":"s1","tool_name":"read","tool_input":{"file_path":"/repo/src/app.py"}}',
    '{"session_id":"s2","tool_name":"read","tool_input":{"file_path":"/repo/.env"}}',
    '{"session_id":"s3","tool_name":"grep","tool_input":{"file_path":"/repo/.env.production"}}',
]

for s in samples:
    code, msg = env_guard(s)
    label = "ALLOW" if code == 0 else "BLOCK"
    print(f"{label} (exit {code}): {msg or '<silent>'}")

> **🏫 During class:**
> 1. Run the cell — first sample allows, second and third block with stderr messages.
> 2. Say: *"Exit code 2 plus a stderr line is the entire blocking protocol. Claude reads that stderr line and adjusts its plan."*
> 3. Live: add a rule that blocks `Bash` calls whose `command` contains a destructive recursive delete. Run a fake payload through. Ask: *"Could the model still get around this?"* — surfaces that hooks are deterministic, but coverage matters.

---
# 11. Implementing a Hook

Walk through one full hook end to end: blocking Claude from ever reading `.env`.

### `.claude/settings.local.json`

```json
{
  "hooks": {
    "preToolUse": [
      {
        "matcher": "read|grep",
        "command": "node ./hooks/read_hook.js"
      }
    ]
  }
}
```

### `hooks/read_hook.js`

```js
let raw = "";
process.stdin.on("data", c => raw += c);
process.stdin.on("end", () => {
  const evt = JSON.parse(raw);
  const path = evt.tool_input?.file_path || evt.tool_input?.path || "";
  if (path.includes(".env")) {
    console.error(`Refusing to read ${path}`);
    process.exit(2);
  }
});
```

### Lessons from doing it for real

- **Restart Claude Code** after editing the settings file — hook configs are read at startup.
- `console.error` (stderr) is the channel Claude reads. `console.log` (stdout) is silent to the model.
- `tool_input.path` vs `tool_input.file_path` — handle both for forward-compatibility.
- Once installed, Claude *recognizes* it was blocked and explains the situation back to the user. That self-awareness only works if you write a clear stderr message.

### Demo: ask Claude to react to its own block message

We craft the message Claude Code would receive after a pre-hook blocks a `.env` read, then ask Claude what it would say to the user. This is exactly the model's job in the real flow: take the stderr feedback and translate it into a useful explanation. Try a vaguer stderr message ("blocked") to see how the explanation degrades — that's the case for writing specific hook messages.

In [ ]:
hook_feedback = (
    "Tool call blocked by pre-tool-use hook. Stderr from hook:\n"
    "  Refusing to read /repo/.env (path contains .env)\n"
    "Explain the situation to the user and propose a safe alternative."
)

print(chat(add_user_message([], hook_feedback), temperature=0.2))

> **🏫 During class:**
> 1. Run the cell — Claude should explain the block and offer something like *"I can read `.env.example` instead, or you can paste the relevant variable."*
> 2. Say: *"This is why the stderr message matters. Claude doesn't know *why* it was blocked unless your hook tells it."*
> 3. Replace the stderr line with a generic *"blocked"* and rerun. Ask: *"Was the user-facing explanation worse? In what way?"*

---
# 12. Useful Hooks

Two hooks worth shipping in any project.

### Hook 1 — Type Checker (post-tool-use)

- **Problem**: Claude edits a function signature but forgets to update one of seven call sites. A compile-time error you only hit when you next run the code.
- **Solution**: After every `*.ts` edit, run `tsc --no-emit`, capture the errors, feed them back as stderr.
- **Outcome**: Claude sees the type error and fixes the call sites in the same turn.
- **Adapt to other stacks**: `mypy`, `pyright`, `cargo check`. For untyped languages, run a fast test subset.

### Hook 2 — Duplicate-Code Detector (post-tool-use, sub-Claude)

- **Problem**: On bigger tasks Claude loses focus and writes a *new* SQL query when one already exists. The codebase grows hair.
- **Solution**: When a file in `queries/` changes, spawn a *second* Claude (via the SDK) whose only job is to compare the new code to the existing siblings. If it finds a duplicate, it exits 2 with a stderr like *"reuse `getUserById` instead of duplicating it."*
- **Trade-off**: Extra time and cost per edit. Apply it only to directories where duplication is expensive.

### The shared pattern

Both hooks are **automated feedback loops**. Run a fast deterministic check, capture the result, and let Claude self-correct on the next turn — instead of catching the issue in code review three days later.

### Demo: a sub-Claude reviewing another Claude's output

We have Claude write a SQL query, then dispatch a *second* call whose system prompt is *"check this query against the canonical list and warn if it duplicates one."* The second call is the sub-Claude a duplicate-detector hook would spawn. Try having the writer produce a brand-new query (no duplicate) and watch the reviewer correctly stay quiet — false-positive resistance is the whole game here.

In [ ]:
EXISTING_QUERIES = (
    "-- get_user_by_id: SELECT * FROM users WHERE id = $1\n"
    "-- list_active_users: SELECT * FROM users WHERE deleted_at IS NULL\n"
)

writer_prompt = (
    "Write a single SQL query (no commentary, no fences) that fetches one user "
    "by their primary key id."
)
new_query = chat(add_user_message([], writer_prompt), temperature=0).strip()
print("Writer produced:", new_query)

reviewer_system = (
    "You are a duplicate-query detector. Compare the candidate query against the "
    "existing queries. If it duplicates one, respond with exactly:\n"
    "  DUPLICATE: <existing_name>\n"
    "Otherwise respond with exactly:\n"
    "  OK"
)
reviewer_prompt = f"Existing queries:\n{EXISTING_QUERIES}\nCandidate:\n{new_query}"
verdict = chat(add_user_message([], reviewer_prompt), system=reviewer_system, temperature=0)
print("Reviewer verdict:", verdict.strip())

> **🏫 During class:**
> 1. Run the cell — the reviewer should print `DUPLICATE: get_user_by_id` because the writer reinvents an existing query.
> 2. Say: *"This is a hook in shape — fast deterministic check, surfaced as feedback. Notice the reviewer's response format is constrained so the hook can parse it without an LLM."*
> 3. Change the writer prompt to *"a query that lists users created in the last 24 hours"* and rerun. Ask: *"Did the reviewer correctly stay silent? If it said DUPLICATE, why was it wrong?"*

---
# 13. The Claude Code SDK

The Claude Code SDK is a programmatic interface — CLI, TypeScript, and Python — that exposes the same tool-using assistant you get in the terminal.

### What it's for

Embedding Claude Code's intelligence into a pipeline you already own. Examples:

- A pre-merge bot that reviews diffs.
- A nightly job that audits dependency changes.
- The hook from §12 that spawns a sub-Claude.

### Defaults

- **Read-only** by default — files, directories, grep. Safe for CI by construction.
- **Write** requires opt-in: pass `allowTools: ["edit", ...]` (or set it in `.claude/`).
- **Output**: a stream of conversation messages between local Claude Code and the model. The final message is the user-facing answer.

### When to reach for it

- Helper scripts inside an existing project — yes.
- A standalone "build me an app from scratch" agent — usually no; the terminal is better.

### The plain-Anthropic-API analogue

The Claude Code SDK is a *higher-level* client: it bundles default tools, file-system access, and the conversation loop. The plain Anthropic SDK gives you the model and lets you build the rest yourself — which is what we do in this notebook.

### Demo: a tiny "Claude Code SDK"-style loop with the plain SDK

We hand-roll a one-step tool-use loop: the model can call `list_files`, we execute it for real (against `os.listdir`), feed the result back, and let the model produce a final answer. This is the entire idea of the Claude Code SDK in twenty lines — a tool registry, an execution loop, a final answer. Add a second tool (e.g., `read_file`) live to see the model chain calls.

In [ ]:
import os, json

def list_files(path: str):
    try:
        return {"entries": sorted(os.listdir(path))[:20]}
    except Exception as e:
        return {"error": str(e)}

sdk_tools = [
    {
        "name": "list_files",
        "description": "List up to 20 entries in a directory.",
        "input_schema": {
            "type": "object",
            "properties": {"path": {"type": "string"}},
            "required": ["path"],
        },
    }
]

messages = [{"role": "user", "content": "What Python files live at the top of this repo? Use list_files for '.'."}]

resp = client.messages.create(model=model, max_tokens=600, tools=sdk_tools, messages=messages)
messages.append({"role": "assistant", "content": resp.content})

if resp.stop_reason == "tool_use":
    tu = next(b for b in resp.content if b.type == "tool_use")
    result = list_files(**tu.input)
    print("[tool result]", json.dumps(result)[:200])
    messages.append({
        "role": "user",
        "content": [{"type": "tool_result", "tool_use_id": tu.id, "content": json.dumps(result)}],
    })
    final = client.messages.create(model=model, max_tokens=400, tools=sdk_tools, messages=messages)
    print("\n[final answer]", final.content[0].text)
else:
    print(resp.content[0].text)

> **🏫 During class:**
> 1. Run the cell — first you'll see the directory listing, then Claude's natural-language summary of which files are Python.
> 2. Say: *"This is the Claude Code SDK in miniature. Tool registry, execution loop, final answer. The real SDK adds defaults, safety, and a thousand tools — same shape."*
> 3. Add a `read_file` tool live and ask: *"What changes in the loop to support it?"* — the answer is *"the loop runs again until `stop_reason != 'tool_use'`,"* which is the one structural lesson worth taking home.

---
## Recap

- A coding assistant = LM + tool-use loop. Claude leads on tool quality, which is what makes Claude Code feel like a teammate.
- Context is the biggest quality lever. `CLAUDE.md` is the durable mechanism; `system` prompts are the API analogue.
- Plan/Think/screenshots/git integration are the day-to-day change-making toolbox.
- `Esc`, `/compact`, `/clear` keep the working set tight — the API analogue is just trimming `messages`.
- Custom commands are parameterized prompt templates. MCP servers are tool providers. GitHub integration is MCP-shaped tools wired to GitHub APIs.
- Hooks turn fuzzy reminders into enforced rules: pre-hooks block, post-hooks check.
- The Claude Code SDK is the programmatic version — same engine, embeddable in pipelines.

## Exercises

1. **Templated review.** Write a `slash_explain(snippet)` template that returns a 3-line plain-English explanation of any code snippet. Use the same `$arguments`-replacement pattern as §6.
2. **Two-step plan-then-execute.** First call: ask Claude for a numbered plan to add a `/healthz` endpoint to a Flask app. Second call: hand the plan back as `system` and ask for the code. Note where the second call gains by having the first one in context.
3. **A real hook payload.** Write a Python function that takes a tool-call payload and blocks any `Bash` command containing a destructive recursive delete, a piped shell installer, or a world-writable chmod. Print the stderr message a real hook would emit.
4. **Sub-agent reviewer.** Extend the §12 demo so the *writer* attempts a query, the *reviewer* gives `OK` or `DUPLICATE`, and on `DUPLICATE` you re-prompt the writer with the reviewer's verdict and re-check. Stop after two rounds.
5. **Hand-rolled SDK.** Add `read_file(path)` to the §13 tool list and turn the single-step loop into a `while resp.stop_reason == "tool_use":` loop. Run *"open `pyproject.toml` and tell me what Python version this targets."*

In [ ]:
# Exercise 1 — slash_explain
# Fill in:
#
# EXPLAIN_TEMPLATE = (
#     "Explain the following code in exactly three plain-English sentences. "
#     "No code, no jargon.\n\n```\n$arguments\n```"
# )
#
# def slash_explain(arguments: str) -> str:
#     ...
#
# print(slash_explain("def fizzbuzz(n):\n    ..."))